In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
import pandas as pd

from model import SocialGPModelSBI
from rewards import (
    sample_children_with_corr,
    make_parent_and_children_cholesky2,
    build_corr_matrix_option1,
    build_corr_matrix_option2,
    build_corr_matrix_option3,
    _min_max,
)
from utils import plot_summary
from tqdm import tqdm

from simulate_reward_curves import parallel_simulate as parallel_simulate_base

seed = 0
rng: np.random.Generator = np.random.default_rng(seed)
corr_matrix = build_corr_matrix_option1()  # parent + 3 children
n_children = corr_matrix.shape[0] - 1

seed = 0
a_rng = np.random.default_rng(seed)

grid_size = 25
parent, child_maps = make_parent_and_children_cholesky2(
    rng=a_rng,
    grid_size=grid_size,
    n_children=n_children,
    length_scale=2.0,
    corr_matrix=corr_matrix,
)

# keep reward scale consistent with other sims
child_maps = [_min_max(c) - 0.5 for c in child_maps]

fig, axes = plt.subplots(1, len(child_maps) + 1, figsize=(4 * (len(child_maps) + 1), 4))

# parent
axes[0].imshow(parent, origin='lower')
axes[0].set_title("Parent")
axes[0].axis('off')

# children
for i, (c, ax) in enumerate(zip(child_maps, axes[1:]), start=1):
    ax.imshow(c, origin='lower')
    ax.set_title(f"Child {i}")
    ax.axis('off')

plt.tight_layout()
plt.show()
m = SocialGPModelSBI(
    child_maps=child_maps,
    rng=a_rng,
    n=n_children,
    length_scale_private=1.11,
    length_scale_social=1.11,
    observation_noise_private=0.0001,
    observation_noise_social=3,
    tau=0.03,
    beta_private=0.33,
    beta_social=0.33,
)

for _ in range(15):
    m.step()

results = m.datacollector.get_model_vars_dataframe()

plot_summary([results])